In [ ]:
%load_ext autoreload
%autoreload 2
from eo_tools.S1.download import download_partial_products, search_products
from eo_tools.util import explore_products
from eo_tools_dev.util import serve_map
import geopandas as gpd
import json
import numpy as np

import logging
logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)
from pathlib import Path
import rasterio
import rioxarray as riox
import xarray as xr


In [ ]:
def percentile_stretch(img, valid=None, percentiles=(2, 98), gamma=1.0):
    img = np.asarray(img, dtype="float32")
    if valid is None:
        valid = np.isfinite(img) & (img != 0)

    out = np.zeros(img.shape, dtype="float32")
    if not np.any(valid):
        return out

    vmin, vmax = np.nanpercentile(img[valid], percentiles)
    if vmax <= vmin:
        return out

    out[valid] = np.clip((img[valid] - vmin) / (vmax - vmin), 0, 1)
    if gamma != 1.0:
        out[valid] = out[valid] ** gamma
    return out

def write_polsar_rgb(in_dir, amp_prefix, out_file):
    in_dir = Path(in_dir)
    vv = riox.open_rasterio(in_dir / f"{amp_prefix}_vv.tif").squeeze("band", drop=True)
    vh = riox.open_rasterio(in_dir / f"{amp_prefix}_vh.tif").squeeze("band", drop=True)

    vv_data = vv.values.astype("float32")
    vh_data = vh.values.astype("float32")
    valid = np.isfinite(vv_data) & np.isfinite(vh_data) & (vv_data > 0) & (vh_data > 0)

    ratio = np.zeros_like(vv_data, dtype="float32")
    ratio[valid] = vh_data[valid] / (vv_data[valid] + 1e-30)

    red = percentile_stretch(np.log1p(vv_data), valid, gamma=0.85)
    green = percentile_stretch(np.log1p(vh_data), valid, gamma=0.85)
    blue = percentile_stretch(ratio, valid, percentiles=(2, 99), gamma=0.75)
    rgb = (255 * np.stack([red, green, blue])).round().astype("uint8")

    arrout = xr.DataArray(
        rgb,
        coords={"band": [1, 2, 3], **{dim: vv.coords[dim] for dim in vv.dims}},
        dims=("band", *vv.dims),
        name="polsar_rgb",
    )
    arrout.rio.write_crs(vv.rio.crs, inplace=True)
    arrout.rio.write_transform(vv.rio.transform(), inplace=True)
    arrout.rio.write_nodata(0, inplace=True)

    out_path = in_dir / out_file
    arrout.rio.to_raster(out_path, driver="COG")
    return out_path


In [ ]:
data_dir = "/data/S1/partial_dls/Andorra_TS/"

output_root = "/data/res/test-andorra-ts"
# use the creds created on CDSE website
with open("/data/creds_s3.json") as f:
    cred = json.load(f)

## Search products with STAC API

In [ ]:
# Search using STAC api
aoi_file = "/eo_tools/data/Andorra_tiny.geojson"
shp = gpd.read_file(aoi_file).geometry[0]


# original products
# search_criteria = {
#     "start": "2018-12-22",
#     "end": "2018-12-29",
#     "geom": shp
# }

# products = search_products(
#     intersects=shp,
#     datetime=["2026-03-01", "2026-04-20"],
# )

In [ ]:
m = explore_products(products=products, aoi=shp)
serve_map(m)

In [ ]:
# sel = products[products.relativeOrbitNumber == 124]

sel = products.iloc[[1,5,8,12,15,19]]

In [ ]:
download_partial_products(sel, shp, out_dir=data_dir ,aws_key=cred["username"], aws_secret=cred["password"], force_overwrite=False)

In [ ]:
partial_products = sorted(Path(data_dir).glob("*.partial.SAFE"))
print(f"Found {len(partial_products)} partial products")
for product_dir in partial_products:
    print(product_dir)

In [ ]:
# let's sort products by date to process them sequentially
sel_sorted = sel.set_index("startTimeFromAscendingNode").sort_index()
sel_sorted.to_csv(Path(data_dir) / "selected_products.csv")

In [ ]:
import pandas as pd
from eo_tools.auxils import get_burst_geometry
from eo_tools.S1.process import process_insar, process_slc

sel_sorted = pd.read_csv(Path(data_dir) / "selected_products.csv")

# len(sel.id[1:]), len(sel.id[:-1])


to_process = tuple(zip(sel_sorted.id[:-1], sel_sorted.id[1:]))

for prm_id, sec_id in to_process:

    prm_path = Path(data_dir) / f"{prm_id}.partial.SAFE"
    sec_path = Path(data_dir) / f"{sec_id}.partial.SAFE"

    insar_dir = process_insar(
        prm_path=str(prm_path),
        sec_path=str(sec_path),
        output_dir=Path(output_root),
        aoi_name=None,
        pol="vv",
        subswaths=["IW1", "IW2", "IW3"],
        write_coherence=True,
        write_interferogram=True,
        write_primary_amplitude=True,
        write_secondary_amplitude=False,
        apply_fast_esd=True,
        dem_upsampling=1.8,
        dem_force_download=False,
        dem_buffer_arc_sec=40,
        boxcar_coherence=[3, 3],
        filter_ifg=True,
        multilook=[1, 4],
        warp_kernel="bicubic",
        cal_type="beta",
        clip_to_shape=True,
        orb_dir="/data/S1_orbits/",
    )

    # out_dir_no_flat = process_slc(
    #     slc_path=str(prm_path),
    #     output_dir=output_root,
    #     aoi_name="no_flat",
    #     shp=shp,
    #     pol="full",
    #     subswaths=["IW1", "IW2", "IW3"],
    #     dem_name="cop-dem-glo-30",
    #     dem_upsampling=1.8,
    #     dem_force_download=False,
    #     dem_buffer_arc_sec=40,
    #     multilook=[1, 4],
    #     warp_kernel="bicubic",
    #     cal_type="beta",
    #     clip_to_shape=True,
    # )

    out_dir_flattened = process_slc(
        slc_path=str(prm_path),
        output_dir=output_root,
        aoi_name=None,
        shp=shp,
        pol="full",
        subswaths=["IW1", "IW2", "IW3"],
        dem_name="cop-dem-glo-30",
        dem_upsampling=1.8,
        dem_force_download=False,
        dem_buffer_arc_sec=40,
        multilook=[1, 4],
        warp_kernel="bicubic",
        cal_type="terrain",
        clip_to_shape=True,
    )
    write_polsar_rgb(in_dir=out_dir_flattened, amp_prefix="amp", out_file="dualpol_rgb.tif")
 

In [ ]:
from matplotlib import colormaps
from eo_tools.S1.process import amplitude, apply_to_patterns_for_pair, apply_to_patterns_for_single, geocode_and_merge_iw

def change_detection(amp_prm_file, amp_sec_file, out_file):
    log.info("Smoothing amplitudes")
    amp_prm = riox.open_rasterio(amp_prm_file, chunks={})[0]#.rolling(x=5, y=5, center=True).median()#.mean()
    amp_sec = riox.open_rasterio(amp_sec_file, chunks={})[0]#.rolling(x=5, y=5, center=True).median()#.mean()
    log.info("Incoherent changes")
    ch = np.log(amp_prm+1e-10) - np.log(amp_sec+1e-10)
    ch = ch.rolling(x=5, y=5, center=True).median()
    ch.rio.to_raster(out_file)

def write_change_rgb(change_file, out_file=None, cmap_name="RdBu_r", q=98):
    change = riox.open_rasterio(change_file).squeeze("band", drop=True)
    data = change.values.astype("float32")
    valid = np.isfinite(data)
    if not np.any(valid):
        raise ValueError(f"No valid pixels found in {change_file}")

    vmax = np.nanpercentile(np.abs(data[valid]), q)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    scaled = np.clip((data + vmax) / (2 * vmax), 0, 1)
    rgba = colormaps[cmap_name](scaled)
    rgb = np.round(255 * np.moveaxis(rgba[..., :3], -1, 0)).astype("uint8")
    rgb[:, ~valid] = 0

    arrout = xr.DataArray(
        rgb,
        coords={"band": [1, 2, 3], **{dim: change.coords[dim] for dim in change.dims}},
        dims=("band", *change.dims),
        name="change_rgb",
    )
    arrout.rio.write_crs(change.rio.crs, inplace=True)
    arrout.rio.write_transform(change.rio.transform(), inplace=True)
    arrout.rio.write_nodata(0, inplace=True)

    out_path = Path(out_file) if out_file else Path(change_file).with_name(f"{Path(change_file).stem}_rgb.tif")
    arrout.rio.to_raster(out_path, driver="COG")
    return out_path, vmax

def write_change_abs_gray(change_file, out_file=None, q=98):
    change = riox.open_rasterio(change_file).squeeze("band", drop=True)
    data = np.abs(change.values.astype("float32"))
    valid = np.isfinite(data)
    if not np.any(valid):
        raise ValueError(f"No valid pixels found in {change_file}")

    vmax = np.nanpercentile(data[valid], q)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    gray = np.zeros_like(data, dtype="uint8")
    gray[valid] = np.round(255 * np.clip(data[valid] / vmax, 0, 1)).astype("uint8")

    arrout = xr.DataArray(
        gray[np.newaxis, ...],
        coords={"band": [1], **{dim: change.coords[dim] for dim in change.dims}},
        dims=("band", *change.dims),
        name="change_abs_gray",
    )
    arrout.rio.write_crs(change.rio.crs, inplace=True)
    arrout.rio.write_transform(change.rio.transform(), inplace=True)
    arrout.rio.write_nodata(0, inplace=True)

    out_path = Path(out_file) if out_file else Path(change_file).with_name(f"{Path(change_file).stem}_abs_gray.tif")
    arrout.rio.to_raster(out_path, driver="COG")
    return out_path, vmax


pair_dirs = sorted(pair_dir.parent for pair_dir in Path(output_root).glob("S1_InSAR_*__*/sar"))

for pair_dir in pair_dirs:
    sar_dir = pair_dir / "sar"
    print(f"Post-processing {pair_dir.name}")

    apply_to_patterns_for_single(
        amplitude,
        out_dir=str(sar_dir),
        in_file_prefix="slc_prm",
        out_file_prefix="amp_prm",
        multilook=[2, 8],
    )

    apply_to_patterns_for_single(
        amplitude,
        out_dir=str(sar_dir),
        in_file_prefix="slc_sec",
        out_file_prefix="amp_sec",
        multilook=[2, 8],
    )

    apply_to_patterns_for_pair(
        change_detection,
        out_dir=str(sar_dir),
        prm_file_prefix="amp_prm",
        sec_file_prefix="amp_sec",
        out_file_prefix="change",
    )

    geocode_and_merge_iw(pair_dir, shp=shp, var_names=["change"])

    for pol in ["vv", "vh"]:
        change_file = pair_dir / f"change_{pol}.tif"
        if change_file.exists():
            rgb_file, vmax = write_change_rgb(change_file, cmap_name="RdBu_r", q=100)
            gray_file, gray_vmax = write_change_abs_gray(change_file)
            print(f"  wrote {rgb_file.name} with symmetric range +/-{vmax:.3f}")
            print(f"  wrote {gray_file.name} for |change| with range 0..{gray_vmax:.3f}")


In [36]:
from eo_tools.dem import retrieve_dem
retrieve_dem(shp, out_file="/data/res/test-andorra-ts/dem.tif", dem_name="cop-dem-glo-30")

INFO:eo_tools.dem:Retrieve DEM (cop-dem-glo-30)
